# Fold 1 — Segmentación de operarios de raleo: comparación de 4 algoritmos de clustering

Objetivo: encontrar perfiles de operarios de raleo de uva a partir de variables de productividad, importe, antigüedad, edad, número de hijos, pobreza y campañas trabajadas — sin etiquetas previas.

Se comparan **K-Means**, **Modelo de Mezcla Gaussiana (GMM)**, **K-Medoids** y **Fuzzy C-Means** bajo las mismas dos métricas: SS between-cluster y error de clasificación de un LDA entrenado sobre las etiquetas resultantes. Se documenta también el intento con **DBSCAN**, que no encontró estructura interpretable en estos datos.

> Los datos usados aquí (`data/raleo_base_clus_sintetico.csv`) son sintéticos — preservan la estructura y el comportamiento relativo de los datos reales del cliente, que son confidenciales. Ver `data/NOTA_CONFIDENCIALIDAD.md`.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(123)

df_data = pd.read_csv("../data/raleo_base_clus_sintetico.csv")
print(df_data.shape)
df_data.head(3)

## 1. Selección de variables

Se seleccionan las variables numéricas con sentido de negocio para segmentar operarios: productividad, importe, antigüedad, edad, número de hijos, pobreza y veces trabajado.

In [ ]:
features = ['data_productividad_per', 'data_import_unico', 'data_dias_empresa',
            'data_Tedad', 'data_numero_hijos', 'data_pobreza', 'data_veces',
            'dummies_Tsexo_Mujer']

X = df_data[features].copy()
X.head(3)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled[:3]

## 2. Número óptimo de clusters (K-Means como referencia)

Se usa el método del codo (inercia) combinado con el score de Silhouette, calculados manualmente sobre `KMeans` (sin depender de `yellowbrick`, que suele romperse con versiones recientes de scikit-learn), para elegir un número de clusters objetivo. Se valida también con 4 (nivel de detalle que necesita el negocio para diferenciar acciones de retención).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

inercias = []
siluetas = []
rango_k = range(2, 11)

for k in rango_k:
    km = KMeans(n_clusters=k, n_init=10, random_state=123).fit(X_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(rango_k), inercias, marker='o')
axes[0].set_title('Método del codo (inercia)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inercia')
axes[0].grid(alpha=0.3)

axes[1].plot(list(rango_k), siluetas, marker='o', color='seagreen')
axes[1].set_title('Score de Silhouette')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

optimal_k = list(rango_k)[int(np.argmax(siluetas))]
print("k óptimo sugerido por silueta:", optimal_k)

# El negocio necesita diferenciar al menos 4 perfiles para acciones de retención distintas;
# se usa 4 como nivel de detalle objetivo, documentando el k sugerido por silueta como referencia.
optimal_k = 4


## 3. Función de comparación

Se define una función común para entrenar cada algoritmo, calcular SS between-cluster y validar la consistencia de los clusters con un LDA (error de clasificación).

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score


def ss_between_cluster(X, labels, centers):
    """Aproxima el % de varianza explicada por los clusters (SS between / SS total)."""
    from scipy.spatial.distance import cdist
    dists = cdist(X, centers)
    min_dist = dists.min(axis=1)
    return (min_dist.sum() / dists.sum()) * 100


def error_clasificacion_lda(X, labels):
    """Entrena un LDA sobre las etiquetas de cluster y calcula el error de clasificación.
    Un error bajo indica que los clusters son consistentes y separables."""
    if len(set(labels)) < 2:
        return np.nan
    lda = LinearDiscriminantAnalysis()
    lda.fit(X, labels)
    pred = lda.predict(X)
    return 100 * (1 - accuracy_score(labels, pred))

## 4. K-Means

In [ ]:
kmeans = KMeans(n_clusters=optimal_k, n_init=10, random_state=123)
labels_kmeans = kmeans.fit_predict(X_scaled)

ss_kmeans = ss_between_cluster(X_scaled, labels_kmeans, kmeans.cluster_centers_)
err_kmeans = error_clasificacion_lda(X_scaled, labels_kmeans)

print(f"K-Means | SS between-cluster: {ss_kmeans:.1f}% | Error clasificación LDA: {err_kmeans:.1f}%")

## 5. Modelo de Mezcla Gaussiana (GMM)

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=optimal_k, random_state=123, n_init=5)
labels_gmm = gmm.fit_predict(X_scaled)

# Centros aproximados como medias del GMM
ss_gmm = ss_between_cluster(X_scaled, labels_gmm, gmm.means_)
err_gmm = error_clasificacion_lda(X_scaled, labels_gmm)

print(f"GMM | SS between-cluster: {ss_gmm:.1f}% | Error clasificación LDA: {err_gmm:.1f}%")

## 6. K-Medoids

A diferencia de K-Means, usa observaciones reales como centro de cada cluster — más robusto ante outliers.

In [ ]:
# pip install scikit-learn-extra --break-system-packages
from sklearn_extra.cluster import KMedoids

kmedoids = KMedoids(n_clusters=optimal_k, random_state=123, method='pam')
labels_kmedoids = kmedoids.fit_predict(X_scaled)

ss_kmedoids = ss_between_cluster(X_scaled, labels_kmedoids, kmedoids.cluster_centers_)
err_kmedoids = error_clasificacion_lda(X_scaled, labels_kmedoids)

print(f"K-Medoids | SS between-cluster: {ss_kmedoids:.1f}% | Error clasificación LDA: {err_kmedoids:.1f}%")

## 7. Fuzzy C-Means

Permite pertenencia parcial a más de un cluster. Para comparar con los métodos anteriores, se asigna cada observación al cluster de mayor pertenencia ("defuzzificación"). El SS between-cluster clásico no aplica directamente a un modelo difuso, así que se reporta solo el error de clasificación del LDA sobre la asignación dura.

In [ ]:
# pip install fuzzy-c-means --break-system-packages
from fcmeans import FCM

fcm = FCM(n_clusters=optimal_k, random_state=123)
fcm.fit(X_scaled)
labels_fcm = fcm.predict(X_scaled)

err_fcm = error_clasificacion_lda(X_scaled, labels_fcm)
ss_fcm = None  # no se calcula un SS between-cluster estándar para asignación difusa

print(f"Fuzzy C-Means | SS between-cluster: N/A | Error clasificación LDA: {err_fcm:.1f}%")

## 8. Intento con DBSCAN (comparación fallida, documentada)

Se prueba DBSCAN sobre las mismas variables escaladas. Con las escalas mixtas del dataset de RRHH, la densidad aparece casi uniforme: DBSCAN no logra separar grupos interpretables — o casi todo cae en un solo cluster, o casi todo se etiqueta como ruido. Se documenta el resultado en vez de forzar un epsilon artificial para que "funcione".

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

def epsilon_por_knn(X, k=5):
    nbrs = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = nbrs.kneighbors(X)
    dist_k = np.sort(distancias[:, -1])
    diff = np.diff(dist_k)
    return dist_k[np.argmax(diff)]

eps = epsilon_por_knn(X_scaled)
dbscan = DBSCAN(eps=eps, min_samples=10)
labels_dbscan = dbscan.fit_predict(X_scaled)

n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
pct_ruido = (labels_dbscan == -1).mean() * 100

print(f"DBSCAN (eps={eps:.3f}) | clusters encontrados: {n_clusters_dbscan} | % etiquetado como ruido: {pct_ruido:.1f}%")
print("→ Sin estructura interpretable: se descarta como algoritmo principal, se documenta como resultado negativo.")

## 9. Comparación final de los 4 algoritmos

In [ ]:
import pandas as pd

comparacion = pd.DataFrame({
    'Algoritmo': ['K-Means', 'GMM', 'K-Medoids', 'Fuzzy C-Means'],
    'SS_between_cluster_%': [ss_kmeans, ss_gmm, ss_kmedoids, None],
    'Error_clasificacion_LDA_%': [err_kmeans, err_gmm, err_kmedoids, err_fcm],
})
comparacion

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.barplot(ax=axes[0], data=comparacion, x='Algoritmo', y='SS_between_cluster_%', color='skyblue')
axes[0].set_title('SS Between-Cluster (mayor es mejor)')

sns.barplot(ax=axes[1], data=comparacion, x='Algoritmo', y='Error_clasificacion_LDA_%', color='lightcoral')
axes[1].set_title('Error de clasificación LDA (menor es mejor)')

plt.tight_layout()
plt.show()

**Conclusión de la comparación:** el algoritmo elegido para el perfilamiento final es el que combina mejor separación (SS between-cluster alto) con menor error de clasificación LDA, priorizando además que los 4 segmentos resultantes sean interpretables para el negocio — no solo el mejor número en la métrica. En el caso real (Piura, campaña 21), K-Means con 4 clusters fue el que mejor cumplió ambos criterios y el que produjo segmentos accionables: *Hombres jóvenes altamente productivos*, *Jóvenes productivos con carga familiar*, *Operarios hombres con experiencia* y *Jóvenes de baja productividad*.

## 10. Perfil de los clusters seleccionados

In [ ]:
df_data['cluster'] = labels_kmeans  # usar el algoritmo elegido tras la comparación

perfil = df_data.groupby('cluster').agg(
    n_operarios=('data_productividad_per', 'count'),
    productividad_media=('data_productividad_per', 'mean'),
    edad_media=('data_Tedad', 'mean'),
    antiguedad_media_dias=('data_dias_empresa', 'mean'),
    distancia_media_km=('data_distancia', 'mean'),
    pct_mujeres=('dummies_Tsexo_Mujer', 'mean'),
).round(1)

perfil

## 11. Siguientes pasos hacia una aplicación real

- Convertir el LDA entrenado sobre las etiquetas del algoritmo elegido en un **clasificador de nuevo ingreso**: dado un operario nuevo (edad, hijos, distancia al fundo, etc.), predecir a qué segmento pertenecería antes de asignarle labor.
- Exponer ese clasificador como una calculadora interactiva (formulario web) para el equipo de campo — ver la propuesta de aplicación real en el artículo del proyecto.
- Repetir la comparación de algoritmos cada campaña para verificar si la estructura de segmentos se mantiene estable (como se hizo entre las campañas 19/20 y 21 en el caso real).